In [15]:
import os
import cv2
import numpy as np
import pytesseract
from glob import glob
from fpdf import FPDF
from PIL import Image
from matplotlib import pyplot as plt

In [8]:
THRESHOLD = 30
COOLDOWN = 10

In [9]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [121]:
def showimg(img):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.imshow(img_rgb)
    plt.title("Image")
    plt.axis("off")
    plt.show()

In [122]:
def get_score(original,duplicate ):
    diff = cv2.absdiff(original, duplicate)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    score = diff.mean()
    return score

def show_diff(original,duplicate ):
    diff = cv2.absdiff(original, duplicate)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    score = diff.mean()
    
    plt.imshow(diff_gray, cmap='gray')
    plt.title("Difference")
    plt.axis("off")
    plt.show()

In [123]:
original = cv2.imread(r"__local__save\1\14.png")
duplicate = cv2.imread(r"__local__save\1\14.png")

In [148]:
video_paths = glob("videos/*")
save_dir = "__DestinctPages__"
vid = None

In [149]:
for path in video_paths:
    print(path)
    name = os.path.splitext(os.path.basename(path))[0]
    print(name)
    save_path = os.path.join(save_dir, name)
    print(save_path)
    vid = path

videos\1.mp4
1
__DestinctPages__\1


In [150]:
cap = cv2.VideoCapture(vid)

In [151]:
prev_gray = None
prev_img = None
page_count = 0
frame_idx = 0
threshold=15
save_next_change = False

array = []

In [152]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if prev_gray is None:
        # First frame → first page
        cv2.imwrite(os.path.join(save_path, f"page_{page_count}.png"), frame)
        page_count += 1
        prev_gray = gray
        prev_img = frame
        continue

    # Compute difference
    diff = cv2.absdiff(prev_gray, gray)
    score = diff.mean()
    array.append(score)
    prev_gray = gray
    
    # If big change → new page
    if score > threshold:
        
        if(save_next_change):
            # showimg(prev_img)
            cv2.imwrite(os.path.join(save_path, f"page_{page_count}.png"), frame)
            print(" score: ",score," for: ",page_count)
        else :
            save_next_change = True
            prev_img = frame
        page_count += 1
        frame_idx += 1
cap.release()

 score:  19.029854600694446  for:  2
